# DART Report Reader — Colab 실행 가이드

DART OpenAPI 인증키 발급: https://opendart.fss.or.kr/uat/uia/egovLoginUsr.do


## 1. 설치

In [ ]:
# GitHub 에서 최신 코드 설치
!pip install git+https://github.com/YOUR_GITHUB_ID/dart-report-reader.git -q
# 또는 로컬 개발 중일 때
# !pip install -e /content/dart-report-reader -q

## 2. API 키 설정

In [ ]:
import os

# 방법 A: 직접 입력 (Colab Secrets 권장)
# from google.colab import userdata
# os.environ['DART_API_KEY'] = userdata.get('DART_API_KEY')

# 방법 B: 직접 입력
os.environ['DART_API_KEY'] = 'YOUR_DART_API_KEY_HERE'

## 3. 설정 & 초기화

In [ ]:
from dart_report_reader import DartReportReader, DartConfig, ReportCode, SectionCode

# 설정 생성
cfg = DartConfig(
    api_key=os.environ['DART_API_KEY'],
    cache_dir='/content/dart_cache',     # 기업 코드 캐시 저장 위치
    output_dir='/content/dart_vault',    # .md 파일 저장 위치
    timeout=30,
    retry=3,
)
print('설정 완료')
print(f'  캐시 경로: {cfg.cache_dir}')
print(f'  출력 경로: {cfg.output_dir}')

In [ ]:
# Reader 생성 & 초기화 (기업 코드 캐시 로드)
reader = DartReportReader(cfg)
reader.init()
print('초기화 완료!')

## 4. 기업 검색

In [ ]:
# 기업명 부분 검색
results = reader.search_corp('삼성')
for r in results[:5]:
    print(f"{r['corp_name']:20s}  종목코드: {r['stock_code']:6s}  고유번호: {r['corp_code']}")

In [ ]:
# 종목코드 → corp_code 변환
corp_code = reader.resolve_corp('005930')
print(f'삼성전자 corp_code: {corp_code}')

## 5. 단일 보고서 파싱

In [ ]:
# 삼성전자 2023 사업보고서에서 배당·임원·직원 섹션만 가져오기
report = reader.fetch_report(
    corp='삼성전자',
    bsns_year=2023,
    reprt_code=ReportCode.ANNUAL,
    sections=[
        SectionCode.DIVIDEND,
        SectionCode.EXECUTIVE,
        SectionCode.EMPLOYEE,
        SectionCode.MAJOR_HOLDER,
        SectionCode.STOCK_TOTAL,
    ],
)

print(f'회사명    : {report.corp_name}')
print(f'사업연도  : {report.bsns_year}')
print(f'보고서    : {report.reprt_label}')
print(f'접수번호  : {report.rcept_no}')
print(f'파싱 섹션 : {[SectionCode.label(s) for s in report.available_sections]}')

In [ ]:
# 배당 데이터 확인
dividend = report.sections.get(SectionCode.DIVIDEND)
if dividend and not dividend.is_empty:
    import pandas as pd
    df = pd.DataFrame(dividend.rows)
    display(df)

## 6. Obsidian Vault 생성

In [ ]:
# 단일 기업 — 여러 연도 일괄 생성
paths = reader.build_vault(
    corp='삼성전자',
    years=[2021, 2022, 2023],
    reprt_codes=[ReportCode.ANNUAL],
    sections=[
        SectionCode.DIVIDEND,
        SectionCode.EXECUTIVE,
        SectionCode.EMPLOYEE,
        SectionCode.MAJOR_HOLDER,
        SectionCode.STOCK_TOTAL,
        SectionCode.BONDS,
        SectionCode.AUDIT_CONTRACT,
    ],
)

print(f'생성된 파일 {len(paths)}개:')
for p in paths:
    print(f'  {p}')

In [ ]:
# 여러 기업 일괄 생성
result = reader.build_vault_multi(
    corps=['삼성전자', '현대자동차', 'NAVER'],
    years=[2023],
    reprt_codes=[ReportCode.ANNUAL],
)

for corp, paths in result.items():
    print(f'{corp}: {len(paths)}개 파일 생성')

## 7. 생성된 md 파일 미리보기

In [ ]:
# 첫 번째 파일 내용 확인
if paths:
    content = paths[0].read_text(encoding='utf-8')
    print(content[:3000])

## 8. 캐시 강제 갱신

In [ ]:
# 기업 코드 목록을 DART 서버에서 새로 받아 캐시 덮어쓰기
reader.refresh_corp_cache()
print('캐시 갱신 완료')

## 9. 생성된 Vault 파일 다운로드 (Colab)


In [ ]:
# ZIP으로 묶어서 다운로드
import shutil
zip_path = '/content/dart_vault.zip'
shutil.make_archive('/content/dart_vault', 'zip', '/content/dart_vault')

from google.colab import files
files.download(zip_path)